In [ ]:
#pip install lime

In [ ]:
# --- Imports ---
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.metrics import mean_squared_error, accuracy_score
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense, BatchNormalization, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import set_random_seed
from tensorflow.keras.losses import Huber

import plotnine as p9
import shap
from lime.lime_tabular import LimeTabularExplainer

pd.options.display.float_format = '{:.6f}'.format

In [ ]:
# Set seed
np.random.seed(1234)

n = 1000
p = 100
rho = 0.5
# Create covariance matrix
s = np.full((p, p), rho)
np.fill_diagonal(s, 1.0)

# Generate multivariate normal data
x = np.random.multivariate_normal(
    mean=np.zeros(p),
    cov=s,
    size=n
)

# Convert to DataFrame with R-like column names V1, V2, ...
x_df = pd.DataFrame(x, columns=[f"V{i}" for i in range(1, p+1)])

# Generate binary variable z
z = np.random.binomial(n=1, p=0.5, size=n)

# Generate noise
noise = np.random.normal(loc=0, scale=0.1, size=n)

# Compute y
y = np.where(
    z == 1,
    3 * (x_df["V1"] + x_df["V2"] + x_df["V3"] + x_df["V4"] + x_df["V5"]) + 3 + noise,
    3 * (x_df["V6"] + x_df["V7"] + x_df["V8"] + x_df["V9"] + x_df["V10"]) - 3 + noise
)

x_df["z"] = z

In [ ]:
dat = x_df.copy()

In [ ]:
dat["y"] = y

In [ ]:
# --- Define MLP model builder ---
def build_mlp(input_dim):
    model = Sequential([
        Input(shape=(input_dim,)),
        Dense(512, activation="relu"),
        BatchNormalization(),
        Dropout(0.2),
        Dense(256, activation="relu"),
        BatchNormalization(),
        Dropout(0.2),
        Dense(128, activation="relu"),
        BatchNormalization(),
        Dropout(0.2),
        Dense(64, activation="relu"),
        BatchNormalization(),
        Dropout(0.2),
        Dense(1, activation="linear")
    ])
    model.compile(optimizer=Adam(learning_rate=0.0005),
                  loss=Huber(delta=1.0))

    return model

In [ ]:
def clique_keras_reg(
    X,
    y,
    model_builder="rf",
    folds=5,
    nsim=26,
    quantile_grid=True,
    keras_model=False,
    epochs=50,
    batch_size=64,
    random_state=123
):
    """
    Fast regression version of CLIQUE with:
      - NumPy-backed computation
      - Stored CV splits
      - Vectorized prediction calls
      - Batched Keras prediction
      - Same return structure as original

    Returns
    -------
    dict:
        {
            "models": list of fitted models,
            "local_imp": pd.DataFrame
        }
    """

    # ============================================================
    # Convert once
    # ============================================================

    X_df = pd.DataFrame(X)
    y_ser = pd.Series(y)

    feature_names = X_df.columns.tolist()

    X_np = X_df.to_numpy(dtype=np.float32)
    y_np = y_ser.to_numpy(dtype=np.float32)

    n, p = X_np.shape

    # ============================================================
    # CV splits (stored once)
    # ============================================================

    cv = KFold(
        n_splits=folds,
        shuffle=True,
        random_state=random_state
    )

    splits = list(cv.split(X_np))

    # ============================================================
    # Train fold models
    # ============================================================

    models = []

    for train_idx, test_idx in splits:

        # ----------------------------------------
        # Build model
        # ----------------------------------------

        if model_builder == "rf":

            model = RandomForestRegressor(
                n_estimators=100,
                random_state=random_state,
                n_jobs=-1
            )

        elif keras_model:

            model = model_builder(p)

        else:

            model = model_builder

        # ----------------------------------------
        # Fit model
        # ----------------------------------------

        if keras_model:

            model.fit(
                X_np[train_idx],
                y_np[train_idx],
                validation_data=(
                    X_np[test_idx],
                    y_np[test_idx]
                ),
                epochs=epochs,
                batch_size=batch_size,
                verbose=0
            )

        else:

            model.fit(
                X_np[train_idx],
                y_np[train_idx]
            )

        models.append(model)

    # ============================================================
    # Base OOF predictions
    # ============================================================

    preds = np.zeros(n, dtype=np.float32)

    for fold, (_, test_idx) in enumerate(splits):

        model = models[fold]

        if keras_model:

            preds[test_idx] = (
                model.predict(
                    X_np[test_idx],
                    batch_size=1024,
                    verbose=0
                ).ravel()
            )

        else:

            preds[test_idx] = model.predict(
                X_np[test_idx]
            )

    # MAE baseline
    base_loss = np.abs(y_np - preds)

    # ============================================================
    # Local importance matrix
    # ============================================================

    local_imp = np.zeros((n, p), dtype=np.float32)

    # ============================================================
    # Main importance loop
    # ============================================================

    for j in range(p):

        col_vals = X_np[:, j]

        # ----------------------------------------
        # Grid
        # ----------------------------------------

        if quantile_grid:

            grid_vals = np.quantile(
                col_vals,
                np.linspace(0, 1, nsim)
            )

        else:

            grid_vals = np.linspace(
                col_vals.min(),
                col_vals.max(),
                nsim
            )

        grid_vals = np.unique(grid_vals)

        # accumulator
        imp_accum = np.zeros(n, dtype=np.float32)

        # ====================================================
        # Fold-specific predictions
        # ====================================================

        for fold, (_, test_idx) in enumerate(splits):

            model = models[fold]

            X_test = X_np[test_idx]

            n_test = len(test_idx)
            n_grid = len(grid_vals)

            # ------------------------------------------------
            # Create giant stacked matrix
            # shape:
            #   (n_grid * n_test, p)
            # ------------------------------------------------

            X_big = np.repeat(
                X_test,
                repeats=n_grid,
                axis=0
            )

            # ------------------------------------------------
            # Replace column j with grid values
            # ------------------------------------------------

            X_big[:, j] = np.tile(grid_vals, n_test)

            # ------------------------------------------------
            # ONE prediction call
            # ------------------------------------------------

            if keras_model:

                pred_big = model.predict(
                    X_big,
                    batch_size=4096,
                    verbose=0
                ).ravel()

            else:

                pred_big = model.predict(X_big)

            # ------------------------------------------------
            # Reshape:
            #   rows = grid vals
            #   cols = observations
            # ------------------------------------------------

            pred_big = pred_big.reshape(n_test, n_grid).T

            # ------------------------------------------------
            # Compute MAE changes
            # ------------------------------------------------

            new_loss = np.abs(
                y_np[test_idx][None, :] - pred_big
            )

            delta = (
                new_loss - base_loss[test_idx][None, :]
            ).mean(axis=0)

            imp_accum[test_idx] = delta

        local_imp[:, j] = imp_accum

        print(f"Finished variable {j+1}/{p}")

    # ============================================================
    # Return same structure as original
    # ============================================================

    local_imp_df = pd.DataFrame(
        local_imp,
        columns=feature_names
    )

    return {
        "models": models,
        "local_imp": local_imp_df
    }

In [ ]:
def ici(model, X, y, feature_idx, nsim = 25):
    """Compute a local importance for one observation and one feature."""
    baseline_pred = model.predict(X).ravel()  # shape (1, n_classes)
    baseline_loss = (y - baseline_pred) ** 2
    
    new_loss = np.zeros((nsim, len(baseline_loss)))

    for i in range(nsim):
        X_permuted = X.copy()
        y_permuted = y.copy()

        other_idx = [i for i in range(X.shape[1]) if i != feature_idx]
        perm = np.random.permutation(X.shape[0])
        X_permuted[:, other_idx] = X[perm][:, other_idx]
        y_permuted = y[perm]
        #X_permuted[:, feature_idx] = np.random.choice(X[:, feature_idx], size=1)
        new_pred = model.predict(X_permuted).ravel()
        new_loss[i, :] = (y_permuted - new_pred) ** 2

    return new_loss.mean(axis=0) - baseline_loss


In [ ]:
res_mlp = clique_keras_reg(x_df, y, model_builder=build_mlp, quantile_grid=True,
                  folds=3, nsim=25, keras_model=True, epochs=100, batch_size=64)

In [ ]:
pd.options.display.float_format = '{:.6f}'.format
res_mlp["local_imp"]

In [ ]:
res_mlp["local_imp"].describe().drop(["count", "mean", "std"])

In [ ]:
gg = (
    p9.ggplot(x_df, mapping=p9.aes(x = x_df["V1"], y = res_mlp["local_imp"]["V1"], color = x_df["z"])) +
    p9.geom_point()
    )

gg

In [ ]:
gg = (
    p9.ggplot(x_df, mapping=p9.aes(x = x_df["z"], y = res_mlp["local_imp"]["V1"], group = x_df["z"])) +
    p9.geom_boxplot()
    )

gg

In [ ]:
gg = (
    p9.ggplot(x_df, mapping=p9.aes(x = x_df["V6"], y = res_mlp["local_imp"]["V6"], color = x_df["z"])) +
    p9.geom_point()
    )

gg

In [ ]:
gg = (
    p9.ggplot(x_df, mapping=p9.aes(x = x_df["z"], y = res_mlp["local_imp"]["V6"], group = x_df["z"])) +
    p9.geom_boxplot()
    )

gg

In [ ]:
Xdf = pd.DataFrame(x_df)
ydf = pd.Series(y)

ann_model = build_mlp(Xdf.shape[1])  # build keras model
kf = KFold(n_splits=3, shuffle=True, random_state=123)
for train_idx, test_idx in kf.split(Xdf):
    ann_model.fit(Xdf.iloc[train_idx], ydf.iloc[train_idx], epochs=100, batch_size=64,
              verbose=0, validation_data=(Xdf.iloc[test_idx], ydf.iloc[test_idx]))

In [ ]:
# explainer_shap = shap.Explainer(ann_model.predict, Xdf.values)
# shap_values = explainer_shap(Xdf.values)

explainer_shap = shap.DeepExplainer(
    ann_model,
    Xdf.values
)
shap_values = explainer_shap.shap_values(Xdf.values)

In [ ]:
shap_values.shape

# Extract SHAP values for the first two variables
shap_v1 = shap_values[:, 0]
shap_v6 = shap_values[:, 5]

In [ ]:
x_lime = x_df.values
p = x_lime.shape[1]

feature_names = [f"Feature_{i}" for i in range(101)]

explainer = LimeTabularExplainer(
    training_data = x_lime,
    feature_names= feature_names,
    mode = "regression"
)

lime_values = np.zeros((len(x_lime), p))

for i in range(len(x_lime)):
    exp = explainer.explain_instance(
        x_lime[i],
        lambda x: ann_model.predict(x, verbose=0).ravel(),
        num_features=p
    )

    # regression uses key 1
    coefs = dict(exp.local_exp[1])

    for j, val in coefs.items():
        lime_values[i, j] = val

In [ ]:
lime_df = pd.DataFrame(lime_values, columns=feature_names)
lime_df

In [ ]:
# Build ici matrix
n_obs, n_feats = x_df.shape
ici_imp = np.zeros((n_obs, n_feats))

for j in range(n_feats):
    ici_imp[:, j] = ici(ann_model, x_lime, y, j, nsim = 25)

ici_df = pd.DataFrame(ici_imp, columns=[f"f{j}" for j in range(n_feats)])

In [ ]:
# Extract LIME values for the first two variables
lime_v1 = lime_df["Feature_0"].values
lime_v6 = lime_df["Feature_5"].values

# --- ICI values ---
ici_v1 = ici_df["f0"].values
ici_v6 = ici_df["f5"].values

In [ ]:
shap_v1 = np.asarray(shap_v1).ravel()
shap_v6 = np.asarray(shap_v6).ravel()

In [ ]:
# Create the first dataframe for the first column of importance values and original values
df_v1 = pd.DataFrame({
    "v1": x_df["V1"],
    "Clique_v1": res_mlp["local_imp"]["V1"],
    "ICI_v1": ici_v1,
    "SHAP_v1": shap_v1,
    "LIME_v1": lime_v1,
    "z": z
})

# Create the second dataframe for the second column of importance values and original values
df_v6 = pd.DataFrame({
    "v6": x_df["V6"],
    "Clique_v6": res_mlp["local_imp"]["V6"],
    "ICI_v6": ici_v6,
    "SHAP_v6": shap_v6,
    "LIME_v6": lime_v6,
    "z": z
})

In [ ]:
# Write dfc to a CSV file
dat.to_csv("df_HD_50_mar.csv", index=False)

# Write df_v1 to a CSV file
df_v1.to_csv("df_v1_HD_50_mar.csv", index=False)

# Write df_v6 to a CSV file
df_v6.to_csv("df_v6_HD_50_mar.csv", index=False)